# Stream the whole planet from one PMTiles file

PMTiles packs an entire tile pyramid into a single static file that is queried with HTTP range requests, so a plain static file host can serve vector tiles without running a tile server. This notebook points MapLibreum at Protomaps' world basemap build — one archive covering the whole planet, from a global view down to individual buildings — and renders it with a handful of hand-written style layers.

**You will use:** `PMTilesSource` and `Map.add_pmtiles_source`. No API key is required; the browser fetches tiles from a public archive when the map is displayed.

In [ ]:
from maplibreum import Map, PMTilesSource, __version__

print(f"MapLibreum {__version__}")

## One archive, the whole planet

[`data.source.coop/protomaps/openstreetmap`](https://source.coop/protomaps/openstreetmap) mirrors Protomaps' basemap build: a single `.pmtiles` file, regenerated from OpenStreetMap, covering every zoom level from the whole globe down to individual buildings. `PMTilesSource` wraps its URL, and `Map.add_pmtiles_source` takes care of loading the PMTiles runtime script and registering the `pmtiles://` protocol before MapLibre starts — the style below just references the archive like any other vector source, using four of its layers (`earth`, `water`, `roads`, `buildings`).

In [ ]:
WORLD_ARCHIVE = "https://data.source.coop/protomaps/openstreetmap/v4.pmtiles"

archive = PMTilesSource(archive_url=WORLD_ARCHIVE)

world_style = {
    "version": 8,
    "sources": {
        "protomaps": {
            "type": "vector",
            "url": archive.style_url,
            "attribution": (
                '<a href="https://protomaps.com">Protomaps</a> © '
                '<a href="https://openstreetmap.org/copyright">OpenStreetMap</a> contributors'
            ),
        }
    },
    "layers": [
        {"id": "background", "type": "background", "paint": {"background-color": "#f7f6f2"}},
        {
            "id": "earth", "type": "fill", "source": "protomaps", "source-layer": "earth",
            "paint": {"fill-color": "#e6e2d3"},
        },
        {
            "id": "water", "type": "fill", "source": "protomaps", "source-layer": "water",
            "paint": {"fill-color": "#a6cee3"},
        },
        {
            "id": "roads", "type": "line", "source": "protomaps", "source-layer": "roads",
            "paint": {
                "line-color": "#b08968",
                "line-width": ["interpolate", ["linear"], ["zoom"], 3, 0.2, 12, 1.6],
            },
        },
        {
            "id": "buildings", "type": "fill", "source": "protomaps", "source-layer": "buildings",
            "paint": {"fill-color": "#c9b896", "fill-opacity": 0.6},
        },
    ],
}

planet = Map(
    map_style=world_style,
    center=[12.4964, 41.9028],
    zoom=4,
    title="One PMTiles file, the whole planet",
    height="580px",
)
planet.add_pmtiles_source(archive)
planet.add_control("navigation", options={"showZoom": True})
planet.add_control("fullscreen")
planet.add_control("scale", position="bottom-left")

planet

## What to remember

- A PMTiles archive is one file; `PMTilesSource` turns its URL into a MapLibre vector source.
- `map.add_pmtiles_source(...)` registers the `pmtiles://` protocol and preloads the archive — no hand-written JavaScript required.
- Style layers reference the archive's `source-layer` names (`earth`, `water`, `roads`, `buildings`, …) exactly like any other vector source.
- The same file serves every zoom level: zoom in on any city in the map above and MapLibre requests only the tiles it needs.

This demo points at a public community mirror of Protomaps' basemap build; for a production deployment, host your own copy (see the [Protomaps basemap downloads guide](https://docs.protomaps.com/basemaps/downloads)) and consider the [`protomaps-themes-base`](https://docs.protomaps.com/basemaps/maplibre) style package for full cartography — labels, land cover, and administrative boundaries — instead of the four hand-written layers used here.

Next: [`opensidewalkmap/`](opensidewalkmap/) shows MapLibreum powering five production-scale applications.